# 02 — Down-weighting

Same pipeline as `01_exclusion.ipynb`, but the top 5% flagged originals (`DOWNWEIGHT_PERCENTILE = 95`) are
**kept** and given `sample_weight = DOWNWEIGHT_FACTOR` (0.2).

**Implementation notes (paper Sec. II.D):**
- The factor is applied in **both** the `WeightedRandomSampler` (class weight x factor) and the training loss
  (`reduction='none'`, weighted mean normalised by the sum of weights). Validation/test losses are unweighted.
- The factor is propagated to augmented copies by filename-stem matching (`<stem>_*`). The **identical
  (unaugmented) copy** stored in the augmented folder has no `_k` suffix, so it is **missed** and keeps weight 1.0.

Before running: see the repo README. Last cell = fold-5 (last fold) diagnostics.


In [ ]:
# CELL 1: INSTALL + IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

!pip install -q scikit-learn scipy

import os, json, random, re, warnings
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from scipy import stats

warnings.filterwarnings('ignore')
print("Imports done.")

import gc

# ── Progress logging (local to the Colab VM) ──────────────────────────────────
LOG_PATH = "/content/outputs/progress_log.csv"
os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)

def log_progress(fold, loop, arch, macro_f1, status="completed"):
    """Append one line per completed training run."""
    row = f"{fold},{loop},{arch},{macro_f1:.4f},{status}\n"
    write_header = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a") as f:
        if write_header:
            f.write("fold,loop,arch,macro_f1,status\n")
        f.write(row)

def get_completed_runs():
    """Read the log and return set of (fold, loop, arch) already done."""
    if not os.path.exists(LOG_PATH):
        return set()
    df = pd.read_csv(LOG_PATH)
    return set(zip(df["fold"], df["loop"], df["arch"]))

completed = get_completed_runs()
print(f"Previously completed runs: {len(completed)}")

In [ ]:

# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 2: CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════


# Download the Malhari dataset from Mendeley Data (DOI 10.17632/m5kxdj7m36.1) and place the
# "colposcopic" folder (CIN1/CIN2/CIN3/<patient folders>/images) here — see README.
DATASET_ROOT = "/content/data/Malhari_Dataset_V1/colposcopic"
AUG_ROOT     = "/content/mendeley_augmented"
OUTPUT_DIR   = "/content/outputs"
AUDIT_DIR    = "/content/audit_loop1"

MERGE_HSIL = False

SEED            = 42
BATCH_SIZE      = 8
IMG_SIZE        = 224
NUM_WORKERS     = 2
MAX_EPOCHS_P1   = 15
MAX_EPOCHS_P2   = 30
LR_P1           = 1e-3
LR_P2           = 1e-4
LABEL_SMOOTHING = 0.15
WEIGHT_DECAY    = 1e-4
DROPOUT         = 0.3
MIXUP_ALPHA     = 0.2
PATIENCE        = 8

ENSEMBLE_ARCHS       = ["resnet18", "mobilenetv3", "efficientnet_b0"]
DST_UNCERTAINTY_MASS = 0.1

K_THRESHOLD          = 0.3
DOWNWEIGHT_PERCENTILE = 95   # top 5% are down-weighted, not excluded (was EXCLUDE_PERCENTILE)
DOWNWEIGHT_FACTOR    = 0.2   # flagged samples keep 20% of their normal sample weight
RELABEL_MIN_AGREE    = 2     # 2-of-3 models must agree (was 3-of-3)
RELABEL_MIN_CONF     = 0.6   # agreeing models must average ≥0.6 confidence

if MERGE_HSIL:
    CLASS_ORDER = ["LSIL", "HSIL"]
    FOLDER_TO_LABEL = {"CIN1": "LSIL", "CIN2": "HSIL", "CIN3": "HSIL"}
else:
    CLASS_ORDER = ["CIN1", "CIN2", "CIN3"]
    FOLDER_TO_LABEL = {"CIN1": "CIN1", "CIN2": "CIN2", "CIN3": "CIN3"}

NUM_CLASSES  = len(CLASS_ORDER)
LABEL_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_LABEL = {i: c for c, i in LABEL_TO_IDX.items()}

def set_seed(s=SEED):
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    np.random.seed(s)
    random.seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | Classes: {CLASS_ORDER}")


In [ ]:
# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 2.5: GENERATE 10× AUGMENTED COPIES (run once)
# ═══════════════════════════════════════════════════════════════════════════════

import os
from PIL import Image, ImageEnhance
from torchvision import transforms
import random
import numpy as np
import torch

mendeley_root = DATASET_ROOT
aug_root = AUG_ROOT
COPIES_PER_IMAGE = 10

def random_augment(img):
    ops = [
        lambda x: transforms.RandomHorizontalFlip(1.0)(x),
        lambda x: transforms.RandomVerticalFlip(1.0)(x),
        lambda x: transforms.RandomRotation(30)(x),
        lambda x: transforms.RandomAffine(0, translate=(0.15, 0.15))(x),
        lambda x: transforms.RandomAffine(0, scale=(0.8, 1.2))(x),
        lambda x: transforms.RandomAffine(0, shear=15)(x),
        lambda x: transforms.RandomPerspective(distortion_scale=0.2, p=1.0)(x),
        lambda x: transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))(x),
        lambda x: ImageEnhance.Brightness(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Contrast(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Sharpness(x).enhance(random.uniform(0.5, 2.0)),
    ]
    n_ops = random.randint(3, 5)
    chosen = random.sample(ops, n_ops)
    for op in chosen:
        img = op(img)
    return img

total = 0
for cin_class in ["CIN1", "CIN2", "CIN3"]:
    class_path = os.path.join(mendeley_root, cin_class)
    for patient in sorted(os.listdir(class_path)):
        patient_path = os.path.join(class_path, patient)
        if not os.path.isdir(patient_path):
            continue
        out_dir = os.path.join(aug_root, cin_class, patient)
        os.makedirs(out_dir, exist_ok=True)
        for fname in sorted(os.listdir(patient_path)):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            img = Image.open(os.path.join(patient_path, fname)).convert("RGB")
            stem = fname.rsplit('.', 1)[0]
            img.save(os.path.join(out_dir, fname))
            for k in range(COPIES_PER_IMAGE):
                random.seed(total * 1000 + k)
                np.random.seed(total * 1000 + k)
                torch.manual_seed(total * 1000 + k)
                aug_img = random_augment(img.copy())
                aug_img.save(os.path.join(out_dir, f"{stem}_{k}.jpg"))
            total += 1

print(f"Done. {COPIES_PER_IMAGE} augmented copies per image.")
print(f"Originals: {total}")
print(f"Total: {total * (1 + COPIES_PER_IMAGE)}")

for cin_class in ["CIN1", "CIN2", "CIN3"]:
    class_path = os.path.join(aug_root, cin_class)
    patients = sorted([p for p in os.listdir(class_path)
                       if os.path.isdir(os.path.join(class_path, p))])
    total_files = sum(len(os.listdir(os.path.join(class_path, p))) for p in patients)
    print(f"  {cin_class}: {len(patients)} patients, {total_files} images")

In [ ]:

# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 3: DATASET DISCOVERY — MENDELEY WITH PATIENT IDS
# ═══════════════════════════════════════════════════════════════════════════════

def discover_mendeley(root):
    """Walk CIN_class/Patient_folder/images structure."""
    records = []
    root = Path(root)
    for cin_class in sorted(os.listdir(root)):
        class_path = root / cin_class
        if not class_path.is_dir() or cin_class not in FOLDER_TO_LABEL:
            continue
        label = FOLDER_TO_LABEL[cin_class]
        label_idx = LABEL_TO_IDX[label]

        for patient_folder in sorted(os.listdir(class_path)):
            patient_path = class_path / patient_folder
            if not patient_path.is_dir():
                continue
            patient_id = f"{cin_class}_{patient_folder.strip()}"

            for fname in sorted(os.listdir(patient_path)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    records.append({
                        "filepath": str(patient_path / fname),
                        "filename": fname,
                        "folder": cin_class,
                        "patient_id": patient_id,
                        "patient_folder": patient_folder,
                        "label": label,
                        "label_idx": label_idx,
                    })
    return pd.DataFrame(records)

 #Originals — used for splitting
df = discover_mendeley(DATASET_ROOT)
n_patients = df["patient_id"].nunique()
print(f"Originals: {len(df)} images from {n_patients} patients")
for label in CLASS_ORDER:
    mask = df["label"] == label
    print(f"  {label}: {mask.sum()} images, {df.loc[mask, 'patient_id'].nunique()} patients")

# Augmented — used to expand training
df_aug = discover_mendeley(AUG_ROOT)
print(f"\nAugmented pool: {len(df_aug)} images from {df_aug['patient_id'].nunique()} patients")


In [ ]:
#Cell 5
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ColposcopyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        # Optional per-sample down-weight multiplier (defaults to 1.0 for all rows
        # if the column isn't present, e.g. val/test sets or pre-audit train sets).
        if "sample_weight" in self.df.columns:
            self.sample_weight = self.df["sample_weight"].astype(float).values
        else:
            self.sample_weight = np.ones(len(self.df), dtype=float)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["label_idx"], row["filepath"], self.sample_weight[idx]

def make_loaders(tr_df, v_df, te_df):
    # All three use eval_transforms — augmentation is pre-computed
    tr_ds = ColposcopyDataset(tr_df, eval_transforms)
    v_ds  = ColposcopyDataset(v_df,  eval_transforms)
    te_ds = ColposcopyDataset(te_df, eval_transforms)

    # Class weights — handle missing classes
    cw = {}
    c = tr_df["label_idx"].value_counts().sort_index()
    for idx in range(NUM_CLASSES):
        cnt = c.get(idx, 1)
        cw[idx] = np.sqrt(len(tr_df) / (NUM_CLASSES * cnt))
    mw = np.mean(list(cw.values()))
    cw = {k: v / mw for k, v in cw.items()}

    # Sampler weight = class-balance weight × per-sample down-weight factor
    # (down-weighted/flagged rows are sampled less often, but never excluded outright)
    sample_w = [cw[idx] for idx in tr_df["label_idx"]]
    if "sample_weight" in tr_df.columns:
        sample_w = [w * dw for w, dw in zip(sample_w, tr_df["sample_weight"])]
    sampler = WeightedRandomSampler(sample_w, len(tr_df), replacement=True)
    g = torch.Generator(); g.manual_seed(SEED)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                           generator=g, drop_last=True)
    v_loader  = DataLoader(v_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    te_loader = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    return tr_loader, v_loader, te_loader, cw

print("Transforms, dataset class, and loader builder ready.")
print("(All splits use eval_transforms — augmentation is pre-computed.)")

In [ ]:

# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 6: PATIENT-LEVEL GROUP K-FOLD
# ═══════════════════════════════════════════════════════════════════════════════
# Real patient IDs from Mendeley. No clustering needed.

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(df, df["label_idx"], groups=df["patient_id"]))

print(f"Group K-Fold: {N_FOLDS} folds on {df['patient_id'].nunique()} patients")
for i, (train_idx, test_idx) in enumerate(folds):
    tr_patients = set(df.iloc[train_idx]["patient_id"])
    te_patients = set(df.iloc[test_idx]["patient_id"])
    leak = tr_patients & te_patients
    te_dist = df.iloc[test_idx]["label"].value_counts().to_dict()
    print(f"  Fold {i+1}: train={len(train_idx)} ({len(tr_patients)} pts) "
          f"test={len(test_idx)} ({len(te_patients)} pts) | "
          f"leakage={len(leak)} | test: {te_dist}")


In [ ]:

# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 7: MODEL ARCHITECTURES
# ═══════════════════════════════════════════════════════════════════════════════


GRADCAM_LAYERS = {
    "resnet18":        "backbone.layer4",
    "mobilenetv3":     "backbone.features.12",
    "efficientnet_b0": "backbone.features.8",
}

class CINClassifier(nn.Module):
    def __init__(self, arch_name="resnet18", num_classes=NUM_CLASSES,
                 dropout=DROPOUT, pretrained=True):
        super().__init__()
        self.arch_name = arch_name

        if arch_name == "resnet18":
            w = models.ResNet18_Weights.DEFAULT if pretrained else None
            base = models.resnet18(weights=w)
            self.feat_dim = base.fc.in_features
            base.fc = nn.Identity()
            self.backbone = base
        elif arch_name == "mobilenetv3":
            w = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
            base = models.mobilenet_v3_small(weights=w)
            self.feat_dim = base.classifier[0].in_features
            base.classifier = nn.Identity()
            self.backbone = base
        elif arch_name == "efficientnet_b0":
            w = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            base = models.efficientnet_b0(weights=w)
            self.feat_dim = base.classifier[1].in_features
            base.classifier = nn.Identity()
            self.backbone = base

        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(self.feat_dim, num_classes))

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats), feats

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

    def get_gradcam_target(self):
        parts = GRADCAM_LAYERS[self.arch_name].split(".")
        layer = self
        for p in parts:
            layer = layer[int(p)] if p.isdigit() else getattr(layer, p)
        return layer

print("CINClassifier defined.")

In [ ]:

# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 8: TRAINING FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════



def mixup_data(x, y, alpha=MIXUP_ALPHA):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    all_p, all_l = [], []
    for images, labels, _, sample_w in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        sample_w = sample_w.to(DEVICE).float()
        mixed, y_a, y_b, lam = mixup_data(images, labels)
        logits, _ = model(mixed)
        # Per-sample loss, scaled by down-weight factor, then averaged.
        # criterion is built with reduction='none' for this purpose.
        per_sample_loss = lam * criterion(logits, y_a) + (1-lam) * criterion(logits, y_b)
        loss = (per_sample_loss * sample_w).sum() / sample_w.sum().clamp_min(1e-8)
        preds = logits.argmax(1)
        correct += (lam * (preds == y_a).float() + (1-lam) * (preds == y_b).float()).sum().item()
        total += labels.size(0)
        all_p.extend(preds.cpu().numpy()); all_l.extend(labels.cpu().numpy())
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
    return total_loss/total, correct/total, f1_score(all_l, all_p, average='macro', zero_division=0)

@torch.no_grad()
def evaluate(model, loader, criterion):
    # Note: evaluate is always called with a plain (reduction='mean') criterion,
    # never the per-sample reduction='none' one used in training.
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_p, all_l, all_prob = [], [], []
    for images, labels, _, _sw in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits, _ = model(images)
        loss = criterion(logits, labels)
        probs = torch.softmax(logits, 1)
        preds = logits.argmax(1)
        total_loss += loss.item() * labels.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_p.extend(preds.cpu().numpy()); all_l.extend(labels.cpu().numpy())
        all_prob.extend(probs.cpu().numpy())
    return (total_loss/total, correct/total,
            f1_score(all_l, all_p, average='macro', zero_division=0),
            np.array(all_p), np.array(all_l), np.array(all_prob))

def train_model(model, tr_loader, v_loader, arch_name, cw):
    save_dir = f"{OUTPUT_DIR}/{arch_name}"
    os.makedirs(save_dir, exist_ok=True)
    wt = torch.tensor([cw[i] for i in range(NUM_CLASSES)], dtype=torch.float32).to(DEVICE)
    # reduction='none' so train_one_epoch can apply per-sample down-weight factors
    # before averaging; eval still uses a plain mean-reduced criterion below.
    criterion = nn.CrossEntropyLoss(weight=wt, label_smoothing=LABEL_SMOOTHING, reduction='none')
    eval_criterion = nn.CrossEntropyLoss(weight=wt, label_smoothing=LABEL_SMOOTHING)
    best_vl, patience_ctr = float('inf'), 0
    history = {"train_loss":[], "val_loss":[], "train_f1":[], "val_f1":[]}

    print(f"\n── P1 WARM-UP [{arch_name}] ──")
    model.freeze_backbone()
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=LR_P1, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS_P1, eta_min=1e-6)

    for ep in range(1, MAX_EPOCHS_P1+1):
        tl, ta, tf = train_one_epoch(model, tr_loader, criterion, opt)
        vl, va, vf, _, _, _ = evaluate(model, v_loader, eval_criterion)
        sched.step()
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_f1"].append(tf); history["val_f1"].append(vf)
        tag = ""
        if vl < best_vl:
            best_vl = vl; patience_ctr = 0
            torch.save(model.state_dict(), f"{save_dir}/best.pt"); tag = " ✅"
        else: patience_ctr += 1
        print(f"  E{ep:02d}/P1 TL:{tl:.4f} TF1:{tf:.3f} | VL:{vl:.4f} VF1:{vf:.3f}{tag}")
        if patience_ctr >= PATIENCE: print("  Early stop"); break

    print(f"\n── P2 FINE-TUNE [{arch_name}] ──")
    model.load_state_dict(torch.load(f"{save_dir}/best.pt", map_location=DEVICE, weights_only=True))
    model.unfreeze_backbone()
    opt = torch.optim.AdamW(model.parameters(), lr=LR_P2, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS_P2, eta_min=1e-7)
    patience_ctr = 0

    for ep in range(1, MAX_EPOCHS_P2+1):
        tl, ta, tf = train_one_epoch(model, tr_loader, criterion, opt)
        vl, va, vf, _, _, _ = evaluate(model, v_loader, eval_criterion)
        sched.step()
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_f1"].append(tf); history["val_f1"].append(vf)
        tag = ""
        if vl < best_vl:
            best_vl = vl; patience_ctr = 0
            torch.save(model.state_dict(), f"{save_dir}/best.pt"); tag = " ✅"
        else: patience_ctr += 1
        print(f"  E{ep:02d}/P2 TL:{tl:.4f} TF1:{tf:.3f} | VL:{vl:.4f} VF1:{vf:.3f}{tag}")
        if patience_ctr >= PATIENCE: print("  Early stop"); break

    model.load_state_dict(torch.load(f"{save_dir}/best.pt", map_location=DEVICE, weights_only=True))
    # Clear GPU cache after training
    torch.cuda.empty_cache()
    gc.collect()

    return history

print("Training functions defined.")


In [ ]:

# %% Cell 9

# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 8: FULL PIPELINE — RUN ALL FOLDS
# ═══════════════════════════════════════════════════════════════════════════════
# ONE cell. Do NOT split. Contains:
#   - DST + XAI helpers
#   - run_fold (Loop 1 → Audit → Loop 2)
#   - Fold loop + aggregation + thesis test

# ── DST helpers ───────────────────────────────────────────────────────────────

def softmax_to_mass(probs, unc=DST_UNCERTAINTY_MASS):
    mass = {}
    remaining = 1.0 - unc
    for i, p in enumerate(probs):
        mass[frozenset({i})] = p * remaining
    mass[frozenset(range(len(probs)))] = unc
    return mass

def dempster_combine(m1, m2):
    combined, K = {}, 0.0
    for A, mA in m1.items():
        for B, mB in m2.items():
            inter = A & B
            prod = mA * mB
            if len(inter) == 0: K += prod
            else: combined[inter] = combined.get(inter, 0.0) + prod
    if K >= 1.0:
        return {frozenset({i}): 1.0/NUM_CLASSES for i in range(NUM_CLASSES)}, K
    for key in combined: combined[key] /= (1.0 - K)
    return combined, K

def combine_multiple(mass_list):
    pairwise_K = {}
    for i, j in combinations(range(len(mass_list)), 2):
        _, k = dempster_combine(mass_list[i], mass_list[j])
        pairwise_K[(i, j)] = k
    result, total_K = mass_list[0], 0.0
    for i in range(1, len(mass_list)):
        result, k = dempster_combine(result, mass_list[i])
        total_K = 1 - (1 - total_K) * (1 - k)
    return result, total_K, pairwise_K

def belief_plausibility(mass):
    res = {}
    for i in range(NUM_CLASSES):
        s = frozenset({i})
        bel = sum(m for A, m in mass.items() if A.issubset(s))
        pl  = sum(m for A, m in mass.items() if len(A & s) > 0)
        res[i] = {"belief": bel, "plausibility": pl}
    return res

# ── XAI helpers ───────────────────────────────────────────────────────────────

class GradCAM:
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        target = model.get_gradcam_target()
        target.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        target.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, image):
        self.model.eval()
        image = image.to(DEVICE).requires_grad_(True)
        logits, _ = self.model(image)
        pred = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, pred].backward(retain_graph=True)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(1, keepdim=True))
        cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() - cam.min() > 1e-8:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)
        return cam, pred

def peripheral_score(heatmap):
    h, w = heatmap.shape
    b = int(min(h, w) * 0.2)
    mask = np.zeros_like(heatmap, dtype=bool)
    mask[:b, :] = mask[-b:, :] = mask[:, :b] = mask[:, -b:] = True
    total = heatmap.sum()
    return float(heatmap[mask].sum() / total) if total > 1e-8 else 0.0

def uniformity_score(heatmap):
    if heatmap.max() < 1e-8: return 1.0
    h = heatmap / (heatmap.sum() + 1e-8)
    h = h.flatten(); h = h[h > 1e-10]
    entropy = -np.sum(h * np.log(h))
    max_ent = np.log(heatmap.size)
    return float(entropy / max_ent) if max_ent > 0 else 1.0

def color_channel_score(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = (img_tensor.cpu() * std + mean).clamp(0, 1)
    ch_means = img.mean(dim=(1, 2)).numpy()
    total = ch_means.sum()
    if total < 1e-8: return 0.0, "none"
    fracs = ch_means / total
    score = (fracs.max() - 1/3) / (2/3)
    return float(score), ["Red", "Green", "Blue"][fracs.argmax()]


# ── Augmented expansion helper ────────────────────────────────────────────────

def expand_with_augmented(fold_train_orig, df_aug):
    """
    Given training originals, pull in their augmented copies from df_aug.
    Matching is by (folder, patient_folder) — guaranteed correct since
    augmented data was generated from Mendeley originals.
    """
    train_patients = set(
        (row["folder"], row["patient_folder"])
        for _, row in fold_train_orig.iterrows()
    )

    def belongs_to_train(row):
        return (row["folder"], row["patient_folder"]) in train_patients

    aug_matches = df_aug[df_aug.apply(belongs_to_train, axis=1)]
    # Remove originals that are already in fold_train_orig (aug folder has copies)
    aug_only = aug_matches[~aug_matches["filepath"].isin(set(fold_train_orig["filepath"]))]

    expanded = pd.concat([fold_train_orig, aug_only], ignore_index=True)
    return expanded, len(aug_only)


# ── run_fold ──────────────────────────────────────────────────────────────────

def run_fold(fold_num, train_idx, test_idx, df, is_last_fold=False):
    print(f"\n{'#'*60}")
    print(f"  FOLD {fold_num + 1}/{N_FOLDS}")
    print(f"{'#'*60}")

    set_seed(SEED + fold_num)

    # ── Split originals: train+val vs test ────────────────────────────────
    fold_trainval = df.iloc[train_idx].reset_index(drop=True)
    fold_test     = df.iloc[test_idx].reset_index(drop=True)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    tv_train_idx, tv_val_idx = next(gss.split(
        fold_trainval, fold_trainval["label_idx"],
        groups=fold_trainval["patient_id"]
    ))
    fold_train = fold_trainval.iloc[tv_train_idx].reset_index(drop=True)
    fold_val   = fold_trainval.iloc[tv_val_idx].reset_index(drop=True)

    # Verify no patient leaks
    tr_pp = set(fold_train["patient_id"])
    va_pp = set(fold_val["patient_id"])
    te_pp = set(fold_test["patient_id"])
    assert len(tr_pp & te_pp) == 0, "Train/test patient leak!"
    assert len(tr_pp & va_pp) == 0, "Train/val patient leak!"
    assert len(va_pp & te_pp) == 0, "Val/test patient leak!"

    print(f"  Train (originals): {len(fold_train)} ({len(tr_pp)} pts) | "
          f"Val: {len(fold_val)} ({len(va_pp)} pts) | "
          f"Test: {len(fold_test)} ({len(te_pp)} pts)")

    # ── Expand training with pre-computed augmented copies ────────────────
    fold_train_expanded, n_aug = expand_with_augmented(fold_train, df_aug)
    print(f"  Train (expanded):  {len(fold_train_expanded)} (+{n_aug} augmented)")

    # ── Build loaders ─────────────────────────────────────────────────────
    tr_loader, v_loader, te_loader, cw = make_loaders(
        fold_train_expanded, fold_val, fold_test
    )

    # ══════════════════════════════════════════════════════════════════════
    #  LOOP 1: Train ensemble (with auto-resume)
    # ══════════════════════════════════════════════════════════════════════
    ens = {}
    for name in ENSEMBLE_ARCHS:
        ens[name] = CINClassifier(arch_name=name).to(DEVICE)

    criterion_eval = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    for arch_name, model in ens.items():
        train_model(model, tr_loader, v_loader, arch_name, cw)
        log_progress(fold_num + 1, 1, arch_name, 0.0)
        torch.cuda.empty_cache()
        gc.collect()

    # Evaluate Loop 1
    all_probs = {}
    for arch_name, model in ens.items():
        _, _, _, _, labels, probs = evaluate(model, te_loader, criterion_eval)
        all_probs[arch_name] = probs

    avg_probs = np.mean(list(all_probs.values()), axis=0)
    ens_preds = avg_probs.argmax(axis=1)
    l1_f1  = f1_score(labels, ens_preds, average='macro', zero_division=0)
    l1_acc = accuracy_score(labels, ens_preds)
    l1_per = [float(f1_score(labels == i, ens_preds == i, zero_division=0))
              for i in range(NUM_CLASSES)]

    print(f"\n  ★ FOLD {fold_num+1} LOOP 1: Macro-F1={l1_f1:.4f} Acc={l1_acc:.3f}")

    # ══════════════════════════════════════════════════════════════════════
    #  AUDIT: DST + XAI on training originals
    # ══════════════════════════════════════════════════════════════════════
    audit_ds = ColposcopyDataset(fold_train, eval_transforms)
    audit_loader = DataLoader(audit_ds, batch_size=32, shuffle=False, num_workers=2)

    arch_names = list(ens.keys())
    for m in ens.values(): m.eval()

    # DST
    dst_results = []
    with torch.no_grad():
        for images, labs_b, fps, _sw in audit_loader:
            images = images.to(DEVICE)
            model_probs = {}
            for aname, model in ens.items():
                logits, _ = model(images)
                model_probs[aname] = torch.softmax(logits, 1).cpu().numpy()

            for i in range(images.size(0)):
                masses = [softmax_to_mass(model_probs[a][i]) for a in arch_names]
                combined, total_K, pw_K = combine_multiple(masses)
                bp = belief_plausibility(combined)
                dst_pred = max(bp, key=lambda k: bp[k]["belief"])
                true_idx = labs_b[i].item()

                per_model_pred = {a: int(model_probs[a][i].argmax()) for a in arch_names}
                per_model_conf = {a: float(model_probs[a][i].max()) for a in arch_names}

                high_k = total_K > K_THRESHOLD
                correct = dst_pred == true_idx
                if high_k and not correct:       quad = "MISLABEL_CANDIDATE"
                elif high_k and correct:         quad = "AMBIGUOUS_CASE"
                elif not high_k and not correct: quad = "MODEL_FAILURE"
                else:                            quad = "CLEAN"

                dst_results.append({
                    "filepath": fps[i], "true_label": IDX_TO_LABEL[true_idx],
                    "conflict_K": total_K, "quadrant": quad,
                    "per_model_pred": per_model_pred,
                    "per_model_conf": per_model_conf,
                })

    # XAI
    best_arch = arch_names[0]
    gradcam = GradCAM(ens[best_arch])

    xai_results = []
    for images, labs_b, fps, _sw in audit_loader:
        for i in range(images.size(0)):
            hm, pred = gradcam.generate(images[i].unsqueeze(0))
            p_s = peripheral_score(hm)
            u_s = uniformity_score(hm)
            c_s, c_d = color_channel_score(images[i])
            artifact = 0.35 * p_s + 0.30 * u_s + 0.35 * c_s
            entry = {"filepath": fps[i], "artifact_score": artifact}
            if is_last_fold:
                entry.update({"heatmap": hm, "peripheral": p_s,
                              "uniformity": u_s, "color_channel": c_s,
                              "dominant_channel": c_d})
            xai_results.append(entry)
    del gradcam
    torch.cuda.empty_cache()
    gc.collect()
    # ── Merge + automated curation ────────────────────────────────────────
    xai_by_path = {r["filepath"]: r for r in xai_results}
    audit = []
    for dst in dst_results:
        xai = xai_by_path.get(dst["filepath"], {})
        artifact = xai.get("artifact_score", 0.0)
        combined = 0.5 * dst["conflict_K"] + 0.5 * artifact
        entry = {
            "filepath": dst["filepath"],
            "true_label": dst["true_label"],
            "conflict_K": dst["conflict_K"],
            "artifact_score": artifact,
            "combined_score": combined,
            "quadrant": dst["quadrant"],
            "per_model_pred": dst["per_model_pred"],
            "per_model_conf": dst["per_model_conf"],
        }
        if is_last_fold:
            entry.update({
                "peripheral": xai.get("peripheral", 0.0),
                "uniformity": xai.get("uniformity", 0.0),
                "color_channel": xai.get("color_channel", 0.0),
                "dominant_channel": xai.get("dominant_channel", ""),
            })
        audit.append(entry)

    # ── Automated curation decisions ──────────────────────────────────────
    # Rule 1 — DOWN-WEIGHT: combined score in top 5% (DOWNWEIGHT_PERCENTILE=95)
    #          Flagged images are NOT dropped from training; instead their
    #          sample weight is scaled by DOWNWEIGHT_FACTOR, reducing their
    #          influence on both the sampler and the loss while keeping
    #          every image (and its augmented copies) in the training set.
    # Rule 2 — RELABEL: ≥2 of 3 models agree on a DIFFERENT class,
    #          with avg confidence of agreeing models ≥ RELABEL_MIN_CONF,
    #          AND low conflict K
    threshold = np.percentile([r["combined_score"] for r in audit], DOWNWEIGHT_PERCENTILE)
    downweight_paths = set()
    relabel_map = {}

    for r in audit:
        preds = list(r["per_model_pred"].values())
        confs = list(r["per_model_conf"].values())
        pred_counts = Counter(preds)
        most_common_pred, most_common_count = pred_counts.most_common(1)[0]

        # Check if majority prediction differs from label
        majority_label = IDX_TO_LABEL[most_common_pred]
        disagrees_with_label = majority_label != r["true_label"]
        meets_agreement = most_common_count >= RELABEL_MIN_AGREE
        low_k = r["conflict_K"] < K_THRESHOLD

        # Confidence of the agreeing models only
        agreeing_confs = [c for p, c in zip(preds, confs) if p == most_common_pred]
        avg_conf = np.mean(agreeing_confs) if agreeing_confs else 0.0
        meets_confidence = avg_conf >= RELABEL_MIN_CONF

        if r["combined_score"] >= threshold:
            r["action"] = "DOWNWEIGHT"
            downweight_paths.add(r["filepath"])
        elif disagrees_with_label and meets_agreement and low_k and meets_confidence:
            r["action"] = "RELABEL"
            r["new_label"] = majority_label
            relabel_map[r["filepath"]] = majority_label
        else:
            r["action"] = "KEEP"

    n_downweight = len(downweight_paths)
    n_relabel = len(relabel_map)
    print(f"  Audit: downweighted={n_downweight} (factor={DOWNWEIGHT_FACTOR}) relabel={n_relabel}")
    if n_relabel > 0:
        for path, new_label in list(relabel_map.items())[:5]:
            orig_label = [r["true_label"] for r in audit if r["filepath"] == path][0]
            print(f"    Relabel: {os.path.basename(path)} {orig_label} → {new_label}")
    # After xai_results and audit list are built, before Loop 2
    del ens          # free all 3 Loop 1 models
    torch.cuda.empty_cache()
    gc.collect()
    # ══════════════════════════════════════════════════════════════════════
    #  LOOP 2: Retrain on curated data (down-weighted, not excluded)
    # ══════════════════════════════════════════════════════════════════════
    curated_train = fold_train.copy()
    # Every original starts at full weight; flagged ones get DOWNWEIGHT_FACTOR.
    curated_train["sample_weight"] = 1.0
    downweight_mask = curated_train["filepath"].isin(downweight_paths)
    curated_train.loc[downweight_mask, "sample_weight"] = DOWNWEIGHT_FACTOR

    for path, new_label in relabel_map.items():
        mask = curated_train["filepath"] == path
        if mask.any():
            curated_train.loc[mask, "label"] = new_label
            curated_train.loc[mask, "label_idx"] = LABEL_TO_IDX[new_label]
    curated_train = curated_train.reset_index(drop=True)

    # Expand curated originals with augmented copies
    curated_expanded, n_aug_curated = expand_with_augmented(curated_train, df_aug)
    # expand_with_augmented pulls columns from df_aug, which has no sample_weight
    # column, so augmented rows default to NaN here — fill those with full weight,
    # then propagate down-weighting to the augmented copies of flagged originals.
    if "sample_weight" not in curated_expanded.columns:
        curated_expanded["sample_weight"] = 1.0
    curated_expanded["sample_weight"] = curated_expanded["sample_weight"].fillna(1.0)

    # Propagate relabels AND down-weighting to augmented copies
    for path, new_label in relabel_map.items():
        # Find the patient folder of this original
        orig_row = fold_train[fold_train["filepath"] == path]
        if orig_row.empty:
            continue
        patient_folder = orig_row.iloc[0]["patient_folder"]
        folder = orig_row.iloc[0]["folder"]
        orig_fname = orig_row.iloc[0]["filename"]
        orig_stem = orig_fname.rsplit('.', 1)[0]

        # Find augmented copies of this specific image
        for idx, row in curated_expanded.iterrows():
            if (row["folder"] == folder and
                row["patient_folder"] == patient_folder and
                row["filename"].startswith(orig_stem + "_")):
                curated_expanded.loc[idx, "label"] = new_label
                curated_expanded.loc[idx, "label_idx"] = LABEL_TO_IDX[new_label]

    for path in downweight_paths:
        orig_row = fold_train[fold_train["filepath"] == path]
        if orig_row.empty:
            continue
        patient_folder = orig_row.iloc[0]["patient_folder"]
        folder = orig_row.iloc[0]["folder"]
        orig_fname = orig_row.iloc[0]["filename"]
        orig_stem = orig_fname.rsplit('.', 1)[0]

        for idx, row in curated_expanded.iterrows():
            if (row["folder"] == folder and
                row["patient_folder"] == patient_folder and
                row["filename"].startswith(orig_stem + "_")):
                curated_expanded.loc[idx, "sample_weight"] = DOWNWEIGHT_FACTOR

    print(f"  Curated: {len(curated_train)} originals → {len(curated_expanded)} expanded "
          f"(downweighted={n_downweight}, relab={n_relabel})")

    set_seed(SEED + fold_num)
    tr_loader_2, v_loader_2, te_loader_2, cw2 = make_loaders(
        curated_expanded, fold_val, fold_test
    )

    ens_2 = {}
    for name in ENSEMBLE_ARCHS:
        ens_2[name] = CINClassifier(arch_name=name).to(DEVICE)

    for arch_name, model in ens_2.items():
        train_model(model, tr_loader_2, v_loader_2, f"{arch_name}_L2", cw2)
        log_progress(fold_num + 1, 2, arch_name, 0.0)
        torch.cuda.empty_cache()
        gc.collect()

    # Evaluate Loop 2
    all_probs_2 = {}
    for arch_name, model in ens_2.items():
        _, _, _, _, labels2, probs2 = evaluate(model, te_loader_2, criterion_eval)
        all_probs_2[arch_name] = probs2

    avg_probs_2 = np.mean(list(all_probs_2.values()), axis=0)
    ens_preds_2 = avg_probs_2.argmax(axis=1)
    l2_f1  = f1_score(labels2, ens_preds_2, average='macro', zero_division=0)
    l2_acc = accuracy_score(labels2, ens_preds_2)
    l2_per = [float(f1_score(labels2 == i, ens_preds_2 == i, zero_division=0))
              for i in range(NUM_CLASSES)]

    delta = l2_f1 - l1_f1
    print(f"  ★ FOLD {fold_num+1} LOOP 2: Macro-F1={l2_f1:.4f} Acc={l2_acc:.3f}")
    print(f"  ★ Δ Macro-F1 = {delta:+.4f}")

    result = {
        "fold": fold_num + 1,
        "loop1": {"macro_f1": l1_f1, "accuracy": l1_acc, "per_class_f1": l1_per},
        "loop2": {"macro_f1": l2_f1, "accuracy": l2_acc, "per_class_f1": l2_per},
        "delta_f1": delta,
        "downweighted": n_downweight,
        "relabeled": n_relabel,
    }

    if is_last_fold:
        result["_viz"] = {
            "audit": audit,
            "xai_results": xai_results,
            "l1_labels": labels, "l1_preds": ens_preds,
            "l2_labels": labels2, "l2_preds": ens_preds_2,
        }

    return result  # <- was missing; every fold previously returned None
 # ══════════════════════════════════════════════════════════════════════════════
#  RUN ALL FOLDS
# ══════════════════════════════════════════════════════════════════════════════

all_fold_results = []
for fold_num, (train_idx, test_idx) in enumerate(folds):
    last = (fold_num == len(folds) - 1)
    result = run_fold(fold_num, train_idx, test_idx, df, is_last_fold=last)
    all_fold_results.append(result)
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  🧹 Fold {fold_num + 1} memory cleared")

# ══════════════════════════════════════════════════════════════════════════════
#  AGGREGATE + THESIS TEST
# ══════════════════════════════════════════════════════════════════════════════

l1_f1s = [r["loop1"]["macro_f1"] for r in all_fold_results]
l2_f1s = [r["loop2"]["macro_f1"] for r in all_fold_results]
deltas = [r["delta_f1"] for r in all_fold_results]
l1_accs = [r["loop1"]["accuracy"] for r in all_fold_results]
l2_accs = [r["loop2"]["accuracy"] for r in all_fold_results]

print(f"\n{'='*60}")
print(f"  {N_FOLDS}-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(f"  Loop 1 (original):  Macro-F1 = {np.mean(l1_f1s):.4f} ± {np.std(l1_f1s):.4f}")
print(f"  Loop 2 (curated):   Macro-F1 = {np.mean(l2_f1s):.4f} ± {np.std(l2_f1s):.4f}")
print(f"  Δ Macro-F1:         {np.mean(deltas):+.4f} ± {np.std(deltas):.4f}")
print(f"  Loop 1 Accuracy:    {np.mean(l1_accs):.4f} ± {np.std(l1_accs):.4f}")
print(f"  Loop 2 Accuracy:    {np.mean(l2_accs):.4f} ± {np.std(l2_accs):.4f}")
print(f"\n  Per-fold breakdown:")
for r in all_fold_results:
    print(f"    Fold {r['fold']}: L1={r['loop1']['macro_f1']:.4f} → "
          f"L2={r['loop2']['macro_f1']:.4f} (Δ={r['delta_f1']:+.4f}) "
          f"[downweighted={r['downweighted']}, relab={r['relabeled']}]")

t_stat, p_value = stats.ttest_rel(l2_f1s, l1_f1s)
print(f"\n  Paired t-test: t={t_stat:.3f}, p={p_value:.4f}")
if p_value < 0.05 and np.mean(deltas) > 0:
    print(f"  ✅ Data auditing significantly improved performance (p<0.05)")
elif np.mean(deltas) > 0:
    print(f"  ⚠ Improvement observed but not statistically significant")
else:
    print(f"  ⚠ Data auditing did not improve performance")
print(f"{'='*60}")

# ── Comparison plot ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
x = np.arange(N_FOLDS); w = 0.35
ax.bar(x - w/2, l1_f1s, w, label="Loop 1 (original)", color='#e74c3c', alpha=0.7)
ax.bar(x + w/2, l2_f1s, w, label="Loop 2 (curated)",  color='#2ecc71', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels([f"F{i+1}" for i in range(N_FOLDS)])
ax.set_ylabel("Macro-F1"); ax.set_title("Per-Fold Macro-F1"); ax.legend()

ax = axes[1]
means = [np.mean(l1_f1s), np.mean(l2_f1s)]
stds  = [np.std(l1_f1s),  np.std(l2_f1s)]
ax.bar(["Loop 1\n(original)", "Loop 2\n(curated)"], means,
       yerr=stds, capsize=5, color=['#e74c3c', '#2ecc71'], alpha=0.7)
ax.set_ylabel("Macro-F1"); ax.set_title("Mean ± Std Across Folds")

ax = axes[2]
ax.text(0.5, 0.65, f"Δ Macro-F1 = {np.mean(deltas):+.4f} ± {np.std(deltas):.4f}",
        ha='center', va='center', fontsize=16,
        color='green' if np.mean(deltas) > 0 else 'red', transform=ax.transAxes)
ax.text(0.5, 0.45, f"p = {p_value:.4f}",
        ha='center', va='center', fontsize=14, transform=ax.transAxes)
ax.text(0.5, 0.25,
        "Statistically significant ✅" if p_value < 0.05 and np.mean(deltas) > 0
        else "Not statistically significant",
        ha='center', va='center', fontsize=12, style='italic', transform=ax.transAxes)
ax.axis('off'); ax.set_title("The Thesis Test")

plt.tight_layout()
os.makedirs(AUDIT_DIR, exist_ok=True)
plt.savefig(f"{AUDIT_DIR}/kfold_comparison.png", dpi=150)
plt.show()

final = {
    "n_folds": N_FOLDS,
    "loop1_f1_mean": float(np.mean(l1_f1s)), "loop1_f1_std": float(np.std(l1_f1s)),
    "loop2_f1_mean": float(np.mean(l2_f1s)), "loop2_f1_std": float(np.std(l2_f1s)),
    "delta_f1_mean": float(np.mean(deltas)), "delta_f1_std": float(np.std(deltas)),
    "p_value": float(p_value), "t_statistic": float(t_stat),
    "per_fold": [{k: v for k, v in r.items() if k != "_viz"} for r in all_fold_results],
}
with open(f"{AUDIT_DIR}/kfold_results.json", "w") as f:
    json.dump(final, f, indent=2)
print(f"Results saved to {AUDIT_DIR}/kfold_results.json")

In [ ]:


# %% ═══════════════════════════════════════════════════════════════════════════
# CELL 12: DIAGNOSTIC PLOTS + GRAD-CAM (from last fold)
# ═══════════════════════════════════════════════════════════════════════════════

viz = all_fold_results[-1]["_viz"]
audit_data = viz["audit"]
xai_data   = viz["xai_results"]
l1_labels, l1_preds = viz["l1_labels"], viz["l1_preds"]
l2_labels, l2_preds = viz["l2_labels"], viz["l2_preds"]
adf = pd.DataFrame(audit_data)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: K vs Artifact Score
ax = axes[0, 0]
cmap_q = {"CLEAN": "#2ecc71", "AMBIGUOUS_CASE": "#f39c12",
          "MODEL_FAILURE": "#e74c3c", "MISLABEL_CANDIDATE": "#9b59b6"}
for q, c in cmap_q.items():
    mask = adf["quadrant"] == q
    ax.scatter(adf.loc[mask, "conflict_K"], adf.loc[mask, "artifact_score"],
               c=c, label=q, alpha=0.5, s=15)
ax.axhline(y=np.percentile(adf["artifact_score"], 90), color='gray', ls='--', alpha=0.4)
ax.axvline(x=K_THRESHOLD, color='gray', ls='--', alpha=0.4)
ax.set_xlabel("Conflict K (DST — label quality)")
ax.set_ylabel("Artifact Score (XAI — image quality)")
ax.set_title("Two Independent Axes of Data Quality")
ax.legend(fontsize=8)

# Plot 2: K by class
ax = axes[0, 1]
for label in CLASS_ORDER:
    mask = adf["true_label"] == label
    ax.hist(adf.loc[mask, "conflict_K"], bins=30, alpha=0.5, label=label)
ax.set_xlabel("Conflict K"); ax.set_ylabel("Count")
ax.set_title("Ensemble Disagreement by Class"); ax.legend()

# Plot 3: Color channel dominance by class
ax = axes[1, 0]
pivot = adf.groupby(["true_label", "dominant_channel"]).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=ax)
ax.set_xlabel("True Label"); ax.set_ylabel("Count")
ax.set_title("Dominant Color Channel by Class\n(detects imaging state mixing)")
ax.tick_params(axis='x', rotation=0)

# Plot 4: Confusion matrices
ax = axes[1, 1]
ax.axis('off')
left = fig.add_axes([0.55, 0.05, 0.18, 0.35])
right = fig.add_axes([0.78, 0.05, 0.18, 0.35])

cm1 = confusion_matrix(l1_labels, l1_preds, labels=range(NUM_CLASSES))
cm2 = confusion_matrix(l2_labels, l2_preds, labels=range(NUM_CLASSES))

left.imshow(cm1, cmap='Reds', interpolation='nearest')
for (i, j), val in np.ndenumerate(cm1):
    left.text(j, i, str(val), ha='center', va='center', fontsize=10)
left.set_xticks(range(NUM_CLASSES)); left.set_yticks(range(NUM_CLASSES))
left.set_xticklabels(CLASS_ORDER, fontsize=7); left.set_yticklabels(CLASS_ORDER, fontsize=7)
left.set_title("Loop 1", fontsize=9)
left.set_ylabel("True", fontsize=8); left.set_xlabel("Pred", fontsize=8)

right.imshow(cm2, cmap='Greens', interpolation='nearest')
for (i, j), val in np.ndenumerate(cm2):
    right.text(j, i, str(val), ha='center', va='center', fontsize=10)
right.set_xticks(range(NUM_CLASSES)); right.set_yticks(range(NUM_CLASSES))
right.set_xticklabels(CLASS_ORDER, fontsize=7); right.set_yticklabels(CLASS_ORDER, fontsize=7)
right.set_title("Loop 2", fontsize=9)
right.set_xlabel("Pred", fontsize=8)

plt.savefig(f"{AUDIT_DIR}/diagnostic_plots.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Diagnostic plots saved to {AUDIT_DIR}/diagnostic_plots.png")

# Save audit report CSV
flat_audit = []
for r in audit_data:
    row = {k: v for k, v in r.items() if k not in ("per_model_pred", "per_model_conf")}
    for mn, p in r.get("per_model_pred", {}).items():
        row[f"pred_{mn}"] = IDX_TO_LABEL[p]
    for mn, c in r.get("per_model_conf", {}).items():
        row[f"conf_{mn}"] = round(c, 4)
    flat_audit.append(row)
pd.DataFrame(flat_audit).to_csv(f"{AUDIT_DIR}/audit_report.csv", index=False)
print(f"Audit report CSV saved to {AUDIT_DIR}/audit_report.csv")

# Grad-CAM overlays for top 20 flagged images
flag_dir = f"{AUDIT_DIR}/flagged_images"
os.makedirs(flag_dir, exist_ok=True)

xai_sorted = sorted([r for r in xai_data if "heatmap" in r],
                     key=lambda r: r["artifact_score"], reverse=True)

for i, r in enumerate(xai_sorted[:20]):
    img = Image.open(r["filepath"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    img_arr = np.array(img) / 255.0
    hm = r["heatmap"]

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(img_arr); axs[0].set_title("Original"); axs[0].axis("off")
    axs[1].imshow(hm, cmap='jet'); axs[1].set_title("Grad-CAM"); axs[1].axis("off")
    overlay = img_arr * 0.5 + plt.cm.jet(hm)[:, :, :3] * 0.5
    axs[2].imshow(overlay); axs[2].set_title("Overlay"); axs[2].axis("off")
    fig.suptitle(f"#{i+1} artifact={r['artifact_score']:.3f} | "
                 f"P={r['peripheral']:.2f} U={r['uniformity']:.2f} "
                 f"C={r['color_channel']:.2f} ({r['dominant_channel']})",
                 fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{flag_dir}/flagged_{i+1:02d}.png", dpi=100)
    plt.close()

print(f"Top 20 flagged Grad-CAM overlays saved to {flag_dir}/")
print("\nDone.")